# Guias de onda e cavidade — teoria vs simulação

- **Frequência de corte TE10:** $f_c = c/(2a)$; abaixo de $f_c$ o modo não propaga.
- **Cavidade rectangular (PEC):** $f_{mnp} = \frac{c}{2}\sqrt{(m/a)^2+(n/b)^2+(p/d)^2}$.
- **Impedância TE:** $Z_{TE} = \eta_0/\sqrt{1-(f_c/f)^2}$ (acima do corte).

Abaixo: (1) cavidade — FFT do campo no centro vs frequências teóricas; (2) corte e Z_TE teóricos.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "emsim").is_dir():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from emsim.fdtd.grid import YeeGrid
from emsim.fdtd.materials import MaterialGrid
from emsim.fdtd.fields import update_E, update_H
from emsim.boundaries.pec import apply_pec
from emsim.sources.gaussian_pulse import GaussianPulse
from Tutorial.common.theory import cavity_frequency, te10_cutoff, te_impedance

## Cavidade rectangular: picos de ressonância vs teoria

In [ ]:
a, b, d = 10e-3, 8e-3, 6e-3
grid = YeeGrid(x_range=(0, a), y_range=(0, b), z_range=(0, d), f0=20e9, resolution=20, courant=0.5)
mat = MaterialGrid(grid.Nz, grid.Ny, grid.Nx, eps_r=1.0, mu_r=1.0, sigma=0.0)
mat.compute_coefficients(grid.dt)

Ex = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))
Ey = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))
Ez = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))
Hx = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))
Hy = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))
Hz = tf.Variable(tf.zeros([grid.Nz, grid.Ny, grid.Nx], dtype=tf.float32))

source = GaussianPulse(f0=20e9, bandwidth=15e9)
ic, jc, kc = grid.Nz // 2, grid.Ny // 2, grid.Nx // 2
Ez_record = []
coeffs = grid.get_curl_coefficients()
inv_dx, inv_dy, inv_dz = coeffs["inv_dx"], coeffs["inv_dy"], coeffs["inv_dz"]
n_steps = 3000
for n in range(n_steps):
    update_H(Ex, Ey, Ez, Hx, Hy, Hz, mat.dt_over_mu, inv_dx, inv_dy, inv_dz)
    update_E(Ex, Ey, Ez, Hx, Hy, Hz, mat.Ca, mat.Cb, inv_dx, inv_dy, inv_dz)
    apply_pec(Ex, Ey, Ez, {"x-", "x+", "y-", "y+", "z-", "z+"})
    if n < 100:
        amp = float(source(n * grid.dt).numpy())
        idx = tf.constant([[ic, jc, kc]], dtype=tf.int32)
        new_val = Ez[ic, jc, kc].numpy() + amp
        Ez.assign(tf.tensor_scatter_nd_update(Ez.read_value(), idx, tf.constant([new_val], dtype=Ez.dtype)))
    Ez_record.append(Ez[ic, jc, kc].numpy())

Ez_record = np.array(Ez_record)
N = len(Ez_record)
fft_vals = fft(Ez_record)
freqs = fftfreq(N, grid.dt)
pos_mask = freqs > 0
freqs_pos = freqs[pos_mask]
fft_mag = np.abs(fft_vals[pos_mask])

In [ ]:
modes = [(1,0,1), (1,1,0), (0,1,1), (2,0,1), (1,1,1)]
f_ana = [cavity_frequency(m, n, p, a, b, d) for m, n, p in modes]

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.plot(freqs_pos / 1e9, fft_mag, label="Simulacao (FFT Ez centro)")
for (m, n, p), f in zip(modes, f_ana):
    ax.axvline(f / 1e9, color="red", alpha=0.7, linestyle="--")
ax.set_xlabel("Frequencia [GHz]")
ax.set_ylabel("|FFT|")
ax.set_title("Cavidade: picos de ressonancia (simulacao) vs frequencias teoricas f_mnp (linhas vermelhas)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 30)
plt.tight_layout()
plt.show()

## Guia rectangular: frequência de corte TE10 e impedância Z_TE(f)

In [ ]:
a_wr42 = 10.67e-3
b_wr42 = 4.32e-3
fc_te10 = te10_cutoff(a_wr42)
print(f"TE10 cutoff (WR42): fc = {fc_te10/1e9:.3f} GHz")

f_ghz = np.linspace(12, 26, 200) * 1e9
Z_te = np.array([te_impedance(1, 0, a_wr42, b_wr42, f) for f in f_ghz])
Z_te_real = np.real(Z_te)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.axvline(fc_te10 / 1e9, color="red", linestyle="--", label=f"fc TE10 = {fc_te10/1e9:.2f} GHz")
ax.plot(f_ghz / 1e9, Z_te_real, label="Z_TE (teoria)")
ax.set_xlabel("Frequencia [GHz]")
ax.set_ylabel("Z_TE [Ohm]")
ax.set_title("Impedancia TE10 e frequencia de corte (guia WR42)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(f_ghz[0]/1e9, f_ghz[-1]/1e9)
plt.tight_layout()
plt.show()